In [ ]:
#!/usr/bin/env python3
"""
RQ2: Passive Detection of Mass PRACH Burst by Eavesdropper
===========================================================
5G NR PRACH eavesdropper detection simulation.
Models whether an eavesdropper with a cheap SDR receiver can detect
mass PRACH (Physical Random Access Channel) transmissions using an
energy detector.

Consolidates: setup.ipynb, path_loss.ipynb, core_detection.ipynb,
              experiment.ipynb, snr_analysis.ipynb
"""

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chi2

1. SETUP — 5G NR PRACH Parameters (from 3GPP TS 38.211)

Preamble Format A1 (common for FR1)
Environment Parameters
Derived quantities

In [ ]:
prach_duration_s = 1e-3          # 1 ms preamble duration
prach_subcarrier_spacing = 30e3  # 30 kHz SCS
prach_bandwidth_hz = 1.08e6      # ~1 MHz PRACH bandwidth
prach_tx_power_dbm = 23          # Max UE transmit power (200 mW)

carrier_freq_hz = 3.5e9          # 3.5 GHz (n78 band, common 5G)
thermal_noise_dbm_per_hz = -174  # Thermal noise floor
eavesdropper_noise_figure_db = 6 # Cheap SDR receiver
boltzmann = 1.38e-23
temperature = 290                # Kelvin

prach_tx_power_w = 10**((prach_tx_power_dbm - 30) / 10)
noise_power_dbm = (thermal_noise_dbm_per_hz
                   + 10 * np.log10(prach_bandwidth_hz)
                   + eavesdropper_noise_figure_db)
noise_power_w = 10**((noise_power_dbm - 30) / 10)

print(f"PRACH TX power: {prach_tx_power_dbm} dBm ({prach_tx_power_w*1e3:.1f} mW)")
print(f"Noise power in PRACH BW: {noise_power_dbm:.1f} dBm ({noise_power_w*1e12:.2f} pW)")

2. PATH LOSS — 3GPP TR 38.901 Urban Macro models

In [ ]:
def path_loss_uma_los_db(distance_m, freq_hz):
    """3GPP TR 38.901 Urban Macro LOS path loss"""
    d_2d = np.asarray(distance_m, dtype=float)
    h_bs = 25    # gNB height (m)
    h_ut = 1.5   # UE height (m)
    d_3d = np.sqrt(d_2d**2 + (h_bs - h_ut)**2)
    fc_ghz = freq_hz / 1e9

    # UMa LOS (valid for 10m < d_2d < 5km)
    d_bp = 4 * h_bs * h_ut * freq_hz / 3e8  # breakpoint distance

    pl = np.where(
        d_2d <= d_bp,
        28.0 + 22 * np.log10(d_3d) + 20 * np.log10(fc_ghz),
        28.0 + 40 * np.log10(d_3d) + 20 * np.log10(fc_ghz)
        - 9 * np.log10(d_bp**2 + (h_bs - h_ut)**2)
    )
    return pl


def path_loss_uma_nlos_db(distance_m, freq_hz):
    """3GPP TR 38.901 Urban Macro NLOS path loss"""
    d_3d = np.sqrt(np.asarray(distance_m, dtype=float)**2 + (25 - 1.5)**2)
    fc_ghz = freq_hz / 1e9
    pl_nlos = 13.54 + 39.08 * np.log10(d_3d) + 20 * np.log10(fc_ghz) - 0.6 * (1.5 - 1.5)
    pl_los = path_loss_uma_los_db(distance_m, freq_hz)
    return np.maximum(pl_nlos, pl_los)


# --- Plot path loss vs distance ---
distances_pl = np.linspace(20, 1500, 500)
plt.figure(figsize=(10, 5))
plt.plot(distances_pl, path_loss_uma_los_db(distances_pl, carrier_freq_hz), label='UMa LOS')
plt.plot(distances_pl, path_loss_uma_nlos_db(distances_pl, carrier_freq_hz),
         label='UMa NLOS', linestyle='--')
plt.xlabel('Distance (m)')
plt.ylabel('Path Loss (dB)')
plt.title('3GPP 38.901 Urban Macro Path Loss at 3.5 GHz')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('rq2_path_loss.png', dpi=150, bbox_inches='tight')
# plt.show()

3. CORE DETECTION — Energy detector model

In [ ]:
def received_energy_from_n_devices(n_devices, distance_m, cell_radius_m,
                                   freq_hz, tx_power_dbm, los=True):
    """
    Model N devices uniformly distributed within cell_radius,
    all transmitting PRACH simultaneously.
    Eavesdropper is at fixed distance from cell center.
    Returns total received power in watts at the eavesdropper.
    """
    # Random UE positions (uniform in disk of cell_radius)
    angles = np.random.uniform(0, 2 * np.pi, n_devices)
    radii = cell_radius_m * np.sqrt(np.random.uniform(0, 1, n_devices))

    # UE positions relative to cell center
    ue_x = radii * np.cos(angles)
    ue_y = radii * np.sin(angles)

    # Eavesdropper position (at 'distance_m' from cell center)
    eav_x, eav_y = distance_m, 0

    # Distance from each UE to eavesdropper
    d_ue_to_eav = np.sqrt((ue_x - eav_x)**2 + (ue_y - eav_y)**2)
    d_ue_to_eav = np.maximum(d_ue_to_eav, 10)  # minimum 10m

    # Path loss for each UE
    if los:
        pl_db = path_loss_uma_los_db(d_ue_to_eav, freq_hz)
    else:
        pl_db = path_loss_uma_nlos_db(d_ue_to_eav, freq_hz)

    # Add shadow fading (log-normal, 4 dB std for LOS, 6 dB for NLOS)
    shadow_std = 4 if los else 6
    shadow_db = np.random.normal(0, shadow_std, n_devices)

    # Received power from each UE
    tx_power_dbm_arr = tx_power_dbm * np.ones(n_devices)
    rx_power_dbm = tx_power_dbm_arr - pl_db - shadow_db
    rx_power_w = 10**((rx_power_dbm - 30) / 10)

    # Total received energy (incoherent addition — powers sum)
    total_rx_power_w = np.sum(rx_power_w)

    return total_rx_power_w


def detection_probability(n_devices, distance_m, cell_radius_m=500,
                          n_trials=2000, pfa=1e-3, los=True):
    """
    Compute probability of detecting N simultaneous PRACH transmissions
    using an energy detector with false alarm probability pfa.
    """
    detections = 0
    snr_values = []

    # Pre-compute threshold (moved OUTSIDE the loop — was a perf bug)
    BT = prach_bandwidth_hz * prach_duration_s  # ~1080
    threshold_factor = chi2.ppf(1 - pfa, 2 * BT) / (2 * BT)

    for _ in range(n_trials):
        # Signal energy
        signal_power = received_energy_from_n_devices(
            n_devices, distance_m, cell_radius_m,
            carrier_freq_hz, prach_tx_power_dbm, los
        )

        # SNR at eavesdropper
        snr_linear = signal_power / noise_power_w
        snr_db = 10 * np.log10(snr_linear + 1e-30)
        snr_values.append(snr_db)

        # Energy detection: signal + noise vs threshold
        test_stat = 1 + snr_linear

        if test_stat > threshold_factor:
            detections += 1

    pd = detections / n_trials
    mean_snr = np.mean(snr_values)
    return pd, mean_snr


print("Detection model ready. Running simulations...")

4. EXPERIMENT — Detection probability vs number of devices

In [ ]:
device_counts = [1, 5, 10, 25, 50, 100, 250, 500, 1000, 5000]
distances = [50, 200, 500, 1000]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for los, ax, title in [(True, axes[0], 'LOS Channel'),
                        (False, axes[1], 'NLOS Channel')]:
    for dist in distances:
        pd_values = []
        for n in device_counts:
            pd, snr = detection_probability(n, dist, n_trials=500, los=los)
            pd_values.append(pd)
            print(f"  {title} | d={dist}m | N={n:5d} | Pd={pd:.3f} | SNR={snr:.1f} dB")

        ax.semilogx(device_counts, pd_values, 'o-', label=f'd = {dist}m')

    ax.set_xlabel('Number of Simultaneous PRACH Transmissions (N)')
    ax.set_ylabel('Detection Probability (Pd)')
    ax.set_title(f'{title} — Pfa = 0.001')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_ylim([-0.05, 1.05])
    ax.axhline(y=0.9, color='red', linestyle=':', alpha=0.5, label='Pd = 0.9 target')

plt.suptitle('RQ2: Passive Detection of Mass PRACH Burst by Eavesdropper\n'
             '3GPP 38.901 UMa | 3.5 GHz | PRACH Format A1 | SDR Receiver (NF=6dB)',
             fontsize=13)
plt.tight_layout()
plt.savefig('rq2_detection_probability.png', dpi=150, bbox_inches='tight')
# plt.show()

print("\nPlot saved to rq2_detection_probability.png")

5. SNR ANALYSIS — SNR vs Number of Devices at various distances

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for los, ls in [(True, '-'), (False, '--')]:
    for dist in [50, 200, 500, 1000]:
        snr_means = []
        for n in device_counts:
            _, snr = detection_probability(n, dist, n_trials=200, los=los)
            snr_means.append(snr)

        label = f'd={dist}m {"LOS" if los else "NLOS"}'
        ax.semilogx(device_counts, snr_means, f'o{ls}', label=label)

ax.set_xlabel('Number of Simultaneous Devices (N)')
ax.set_ylabel('Mean SNR at Eavesdropper (dB)')
ax.set_title('Received SNR from Mass PRACH Burst')
ax.legend(ncol=2, fontsize=9)
ax.grid(True, alpha=0.3)
ax.axhline(y=0, color='red', linestyle=':', alpha=0.5)
ax.annotate('Detection threshold (0 dB)', xy=(5, 0.5), color='red', fontsize=9)

plt.tight_layout()
plt.savefig('rq2_snr_analysis.png', dpi=150, bbox_inches='tight')
# plt.show() # Commented out to prevent blocking in automated runs

6. ADVANCED MODEL — Background Traffic Noise (Poisson Process)

In [ ]:
def detection_probability_bg(n_devices, distance_m, cell_radius_m=500,
                             n_trials=1000, pfa=1e-3, los=True, lambda_bg=0.5):
    """
    Advanced model incorporating random background PRACH traffic
    lambda_bg: average number of normal UEs transmitting in the same PRACH occasion
    """
    # 1. Compute empirical threshold under H0 (Background + Noise Only)
    n_trials_h0 = int(5 / pfa)  # Enough trials for the tail
    h0_stats = np.zeros(n_trials_h0)
    BT = prach_bandwidth_hz * prach_duration_s

    for i in range(n_trials_h0):
        n_bg = np.random.poisson(lambda_bg)
        bg_power = 0
        if n_bg > 0:
            bg_power = received_energy_from_n_devices(
                n_bg, distance_m, cell_radius_m, carrier_freq_hz, prach_tx_power_dbm, los)
        snr_linear = bg_power / noise_power_w
        noise_stat = np.random.chisquare(2 * BT) / (2 * BT)
        h0_stats[i] = noise_stat + snr_linear

    threshold = np.percentile(h0_stats, 100 * (1 - pfa))

    # 2. Compute detection under H1 (Target + Background + Noise)
    detections = 0
    for _ in range(n_trials):
        target_power = received_energy_from_n_devices(
            n_devices, distance_m, cell_radius_m, carrier_freq_hz, prach_tx_power_dbm, los)

        n_bg = np.random.poisson(lambda_bg)
        bg_power = 0
        if n_bg > 0:
            bg_power = received_energy_from_n_devices(
                n_bg, distance_m, cell_radius_m, carrier_freq_hz, prach_tx_power_dbm, los)

        snr_linear = (target_power + bg_power) / noise_power_w
        noise_stat = np.random.chisquare(2 * BT) / (2 * BT)
        test_stat = noise_stat + snr_linear

        if test_stat > threshold:
            detections += 1

    return detections / n_trials

print("Running background traffic model simulation...")
device_counts_bg = [1, 5, 10, 25, 50, 100, 250, 500, 1000]
fig_bg, ax_bg = plt.subplots(1, 2, figsize=(16, 6))

for los, ax_plot, title in [(True, ax_bg[0], 'LOS Channel'),
                       (False, ax_bg[1], 'NLOS Channel')]:
    for dist in [200, 500, 1000]:
        pd_values_bg = []
        for n in device_counts_bg:
            # Lambda = 0.5 implies 1 background UE every 2 ms (extremely busy cell)
            pd_bg = detection_probability_bg(n, dist, n_trials=500, los=los, lambda_bg=0.5)
            pd_values_bg.append(pd_bg)
            # print(f"  {title} | BG \lambda=0.5 | d={dist}m | N={n:5d} | Pd={pd_bg:.3f}")

        ax_plot.semilogx(device_counts_bg, pd_values_bg, 'o-', label=f'd = {dist}m')

    ax_plot.set_xlabel('Number of Simultaneous PRACH Transmissions (N)')
    ax_plot.set_ylabel('Detection Probability (Pd)')
    ax_plot.set_title(f'{title} (Background \u03bb=0.5/occ, Pfa=0.001)')
    ax_plot.legend()
    ax_plot.grid(True, alpha=0.3)
    ax_plot.set_ylim([-0.05, 1.05])
    ax_plot.axhline(y=0.9, color='red', linestyle=':', alpha=0.5, label='Pd = 0.9 target')

plt.suptitle('RQ2: Detection of Mass PRACH Burst with Realistic Background Traffic', fontsize=13)
plt.tight_layout()
plt.savefig('rq2_bg_detection.png', dpi=150, bbox_inches='tight')
# plt.show() # Commented out to prevent blocking in automated runs

print("\nAll simulations complete.")